In [ ]:
# -- Local project setup -------------------------------------------------------
# Run this notebook from anywhere inside the project folder. It will find the
# folder containing Codes, Datasets, and Models automatically.

from pathlib import Path


def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "Codes").is_dir() and (candidate / "Datasets").is_dir() and (candidate / "Models").is_dir():
            return candidate
    raise FileNotFoundError("Could not find project root containing Codes, Datasets, and Models")


PROJECT_ROOT = find_project_root()
DATASETS = PROJECT_ROOT / "Datasets"

print("Project root:", PROJECT_ROOT)


In [ ]:
# -- Beach configuration -------------------------------------------------------
# Manual choice: set BEACH to "bass" or "bicentennial" before running.

BEACH = "bass"

CONFIG = {
    "bass": {
        "CLIPPED_FOLDER": DATASETS / "BCLIPPED",
        "METADATA_FOLDER": DATASETS / "Metadata" / "Bass Metadata",
        "OUTPUT_FOLDER": DATASETS / "Extracted" / "Bass CSV",
    },
    "bicentennial": {
        "CLIPPED_FOLDER": DATASETS / "BICLIPPED",
        "METADATA_FOLDER": DATASETS / "Metadata" / "Bicent Metedata",
        "OUTPUT_FOLDER": DATASETS / "Extracted" / "Bicent CSV",
    },
}

if BEACH not in CONFIG:
    raise ValueError(f"Unknown BEACH {BEACH!r}. Choose one of: {', '.join(CONFIG)}")

CLIPPED_FOLDER = CONFIG[BEACH]["CLIPPED_FOLDER"]
METADATA_FOLDER = CONFIG[BEACH]["METADATA_FOLDER"]
OUTPUT_FOLDER = CONFIG[BEACH]["OUTPUT_FOLDER"]

OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

for label, path in [("CLIPPED_FOLDER", CLIPPED_FOLDER), ("METADATA_FOLDER", METADATA_FOLDER)]:
    if not path.exists():
        raise FileNotFoundError(f"{label} does not exist: {path}")

print("Beach:", BEACH)
print(f"Clipped folder  : {CLIPPED_FOLDER}")
print(f"Metadata folder : {METADATA_FOLDER}")
print(f"Output folder   : {OUTPUT_FOLDER}")


In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import os
import csv
import numpy as np
import rasterio
import xml.etree.ElementTree as ET
from datetime import datetime

In [ ]:
# ── Helper functions ─────────────────────────────────────────────────────────

NS = {
    'gmd': 'http://www.isotc211.org/2005/gmd',
    'gco': 'http://www.isotc211.org/2005/gco',
    'gml': 'http://www.opengis.net/gml/3.2',
    'gmi': 'http://www.isotc211.org/2005/gmi',
    'gmx': 'http://www.isotc211.org/2005/gmx',
}

FT_TO_M = 0.3048006096012192


def parse_grid_metadata(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    def find_text(path):
        el = root.find(path, NS)
        return el.text.strip() if el is not None and el.text else None

    title     = find_text('.//gmd:citation/gmd:CI_Citation/gmd:title/gco:CharacterString')
    anchor    = root.find('.//gmx:Anchor', NS)
    inport_id = anchor.text.strip() if anchor is not None else None
    begin_str = find_text('.//gml:beginPosition')
    end_str   = find_text('.//gml:endPosition')

    begin_date = datetime.strptime(begin_str, '%Y-%m-%d') if begin_str else None
    end_date   = datetime.strptime(end_str,   '%Y-%m-%d') if end_str   else None

    if begin_date and end_date:
        mid_date = begin_date + (end_date - begin_date) / 2
    elif begin_date:
        mid_date = begin_date
    else:
        mid_date = None

    return {
        'survey_title':           title,
        'inport_id':              inport_id,
        'acquisition_date_start': begin_str,
        'acquisition_date_end':   end_str,
        'acquisition_date_mid':   mid_date.strftime('%Y-%m-%d') if mid_date else None,
        'mid_date_obj':           mid_date,
    }


def to_decimal_year(dt):
    if dt is None:
        return None
    year_start  = datetime(dt.year, 1, 1)
    year_end    = datetime(dt.year + 1, 1, 1)
    year_len    = (year_end - year_start).days
    day_of_year = (dt - year_start).days
    return round(dt.year + day_of_year / year_len, 6)


def find_paired_xml(tif_path):
    tif_name  = os.path.basename(tif_path)
    tile_stem = tif_name.replace('_clipped.tif', '').replace('.tif', '')
    j_stem    = tile_stem.split('tR')[0].split('_C')[0]

    for f in os.listdir(METADATA_FOLDER):
        if j_stem in f and 'grid_metadata' in f and f.endswith('.xml'):
            return os.path.join(METADATA_FOLDER, f)
        if j_stem in f and f.endswith('_metadata.xml') and 'aux' not in f:
            return os.path.join(METADATA_FOLDER, f)

    return None


print('Helper functions loaded.')

In [ ]:
# ── Output columns ───────────────────────────────────────────────────────────

FIELDNAMES = [
    'x_m',
    'y_m',
    'elevation_m',
    'x_ft',
    'y_ft',
    'elevation_ft',
    'acquisition_date_start',
    'acquisition_date_end',
    'acquisition_date_mid',
    'decimal_year',
    'survey_title',
    'inport_id',
    'source_file',
]

print(f'{len(FIELDNAMES)} columns defined.')

In [ ]:
# ── Main extraction loop ─────────────────────────────────────────────────────

all_records    = []
survey_summary = []

tif_files = sorted([
    os.path.join(CLIPPED_FOLDER, f)
    for f in os.listdir(CLIPPED_FOLDER)
    if f.endswith('_clipped.tif')
])

if not tif_files:
    print('No _clipped.tif files found. Check your CLIPPED_FOLDER path.')
else:
    print(f'Found {len(tif_files)} clipped TIF files.\n')

# Preflight metadata scan: duplicate midpoint dates usually indicate alternate
# DEM/topobathy products for the same NOAA epoch. Stop so the duplicate product
# can be removed intentionally before extraction and training.
tif_metadata = {}
date_to_files = {}

for tif_path in tif_files:
    tif_name = os.path.basename(tif_path)
    grid_xml = find_paired_xml(tif_path)

    if grid_xml is None:
        print(f'  [WARN] No metadata XML found — skipping {tif_name}')
        continue

    meta     = parse_grid_metadata(grid_xml)
    dec_year = to_decimal_year(meta['mid_date_obj'])
    tif_metadata[tif_path] = (grid_xml, meta, dec_year)

    mid_date = meta['acquisition_date_mid']
    if mid_date:
        date_to_files.setdefault(mid_date, []).append(tif_name)

duplicate_dates = {
    date: files
    for date, files in date_to_files.items()
    if len(files) > 1
}

if duplicate_dates:
    lines = [
        'Duplicate NOAA acquisition midpoint date(s) found.',
        'Remove the duplicate DEM/alternate product or document a manual choice before rerunning.',
    ]
    for date, files in sorted(duplicate_dates.items()):
        lines.append(f'  {date}:')
        for file_name in sorted(files):
            lines.append(f'    - {file_name}')
    raise ValueError('\n'.join(lines))

for tif_path in tif_files:
    tif_name = os.path.basename(tif_path)
    if tif_path not in tif_metadata:
        continue

    grid_xml, meta, dec_year = tif_metadata[tif_path]
    print(f'Processing: {tif_name}')
    print(f'  XML : {os.path.basename(grid_xml)}')

    print(f'  Survey : {meta["survey_title"]}')
    print(f'  Dates  : {meta["acquisition_date_start"]} → {meta["acquisition_date_end"]}')
    print(f'  Mid    : {meta["acquisition_date_mid"]}  (decimal year: {dec_year})')

    with rasterio.open(tif_path) as src:
        data      = src.read(1)
        nodata    = src.nodata
        transform = src.transform
        height, width = data.shape

        records = []
        for row in range(height):
            for col in range(width):
                elev_ft = data[row, col]

                if np.isnan(elev_ft):
                    continue
                if nodata is not None and elev_ft == nodata:
                    continue

                x_ft, y_ft = rasterio.transform.xy(transform, row, col)

                records.append({
                    'x_m':                    round(x_ft    * FT_TO_M, 4),
                    'y_m':                    round(y_ft    * FT_TO_M, 4),
                    'elevation_m':            round(float(elev_ft) * FT_TO_M, 4),
                    'x_ft':                   round(x_ft,   4),
                    'y_ft':                   round(y_ft,   4),
                    'elevation_ft':           round(float(elev_ft), 4),
                    'acquisition_date_start': meta['acquisition_date_start'],
                    'acquisition_date_end':   meta['acquisition_date_end'],
                    'acquisition_date_mid':   meta['acquisition_date_mid'],
                    'decimal_year':           dec_year,
                    'survey_title':           meta['survey_title'],
                    'inport_id':              meta['inport_id'],
                    'source_file':            tif_name,
                })

    out_name = tif_name.replace('_clipped.tif', '_points.csv')
    out_path = os.path.join(OUTPUT_FOLDER, out_name)
    with open(out_path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
        writer.writeheader()
        writer.writerows(records)

    print(f'  Points : {len(records):,}  →  {out_name}\n')

    all_records.extend(records)
    survey_summary.append({
        'file':         tif_name,
        'survey':       meta['survey_title'],
        'date_start':   meta['acquisition_date_start'],
        'date_end':     meta['acquisition_date_end'],
        'date_mid':     meta['acquisition_date_mid'],
        'decimal_year': dec_year,
        'points':       len(records),
    })

print(f'Total points extracted: {len(all_records):,}')

In [ ]:
# ── Save combined CSV ────────────────────────────────────────────────────────

combined_path = os.path.join(OUTPUT_FOLDER, 'NOAA_all_surveys_combined.csv')
with open(combined_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
    writer.writeheader()
    writer.writerows(all_records)

print(f'Combined CSV → {combined_path}')
print(f'Total points : {len(all_records):,}')

In [ ]:
# ── Survey summary table ────────────────────────────────────────────────────

print(f'{"Survey":<60} {"Start":>12} {"End":>12} {"Mid":>12} {"Dec Year":>10} {"Points":>10}')
print('-' * 120)
total = 0
for s in sorted(survey_summary, key=lambda x: x['decimal_year'] or 0):
    title = (s['survey'][:57] + '...') if s['survey'] and len(s['survey']) > 60 else (s['survey'] or 'Unknown')
    print(f"{title:<60} {str(s['date_start']):>12} {str(s['date_end']):>12} "
          f"{str(s['date_mid']):>12} {str(s['decimal_year']):>10} {s['points']:>10,}")
    total += s['points']
print('-' * 120)
print(f'{"TOTAL":<60} {"":>12} {"":>12} {"":>12} {"":>10} {total:>10,}')

In [ ]:
# ── Sanity check ─────────────────────────────────────────────────────────────

with open(combined_path) as f:
    rows = list(csv.DictReader(f))

print(f'Rows    : {len(rows):,}')
print(f'Columns : {len(rows[0])}')
print('\nFirst row:')
for k, v in rows[0].items():
    print(f'  {k:<30}: {v}')